Project Idea -> Consume SpaceX Rockets Api and use Pydantic to build a data models with validation

Utilization data in api json
    -description -> OK
    -cost_per_launch -> OK
    -company -> OK
    -success_rate_pct -> OK
    -height -> OK
    -country -> OK
    -first_flight -> OK
    -rocket_name -> OK

In [85]:
#imports
import requests
from pydantic import BaseModel, Field,model_validator
from pydantic_settings import BaseSettings,SettingsConfigDict
from dotenv import load_dotenv
import os
from typing import List

In [86]:
#request
load_dotenv(dotenv_path="env_practice")
response_data = requests.get(os.environ["API_SPACEX_ROCKET"])

In [ ]:
#class model
class Dimension(BaseModel):
    meters: float
    feet: float
    
    @model_validator(mode="before")
    @classmethod
    def validate_type(cls, values):
        if not isinstance(values['meters'], float):
            values['meters'] = float(values['meters'])
        if not isinstance(values['feet'], float):
            values['feet'] = float(values['feet'])    
        return values

class PayloadWeights(BaseModel):
    id: str
    name: str
    kg: int
    lb: int
    
class Details(BaseModel):
    country: str
    first_flight: str
    cost_per_launch: float
    rocket_name: str
    success_rate_pct: float
    description: str
    
    @model_validator(mode="before")
    def validate_type(cls, values):
        if not isinstance(values['cost_per_launch'], float):
            values['cost_per_launch'] = float(values['cost_per_launch'])
        if not isinstance(values['success_rate_pct'], float):
            values['success_rate_pct'] = float(values['success_rate_pct'])
        return values

class Data(BaseModel):
    company: str = Field(alias="company")
    height: Dimension
    diameter: Dimension
    payload_weights: List[PayloadWeights]
    details: Details = None

In [ ]:
data = Data(**response_data.json())

details_init = Details(country=response_data.json().get('country'), cost_per_launch=response_data.json().get('cost_per_launch'),success_rate_pct=response_data.json().get('success_rate_pct'),first_flight=response_data.json().get('first_flight'), rocket_name=response_data.json().get('rocket_name'), description=response_data.json().get('description'))

data.details = details_init

print(data.model_dump_json())